# 5.3 Estimation Theory

**Goal:** evaluate estimators by their target, bias, consistency, efficiency, and mean squared error.

**Prerequisites:** Modules 0–4 and earlier notebooks in this module; advanced notebooks also build on earlier modules 5–8. Read the section assumptions before using a method.

**Runtime:** Python 3.10+ with NumPy, SciPy, and Matplotlib. Install in your chosen environment with `python3 -m pip install numpy scipy matplotlib`, then open in JupyterLab or VS Code with a Python kernel. Use **Restart Kernel and Run All**. Each notebook is self-contained, includes saved outputs, uses synthetic data and a fixed local random seed, and needs no downloads. Numerical simulations illustrate results; they do not prove them or validate real-world assumptions.

## Module navigation

1. [Sampling Techniques](1_Sampling-Techniques.ipynb)
2. [Sampling Concepts](2_Sampling-Concepts.ipynb)
3. [Estimation Theory](3_Estimation-Theory.ipynb)
4. [Confidence Intervals](4_Confidence-Intervals.ipynb)
5. [Hypothesis Testing](5_Hypothesis-Testing.ipynb)
6. [Statistical Errors](6_Statistical-Errors.ipynb)
7. [Statistical Tests](7_Statistical-Tests.ipynb)

## Learning route

Read each definition and assumption, work through its calculation, run the code, then attempt the practice before viewing the solutions. Distinguish descriptive association, causal inference, and predictive evaluation throughout.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize, special, linalg
rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.titlesize": 12, "font.size": 10})

## 1. Point Estimation and Population Parameter Estimation

A parameter is a population/model quantity such as μ, p, or σ². An estimator is a rule applied to random data; its realized value is an estimate. The sample mean estimates μ and a Bernoulli sample proportion estimates p under the stated sampling model. The target **estimand** must be specified: a user-average loss differs from a session-average loss when users contribute different numbers of sessions.

A statistic computed from selected cases is not automatically an estimator for the entire deployment population. Define the population, unit, weighting, measurement window, and missing-data treatment before evaluating the formula.

**Example:** 8 successes in 10 IID Bernoulli trials gives p-hat=0.8. It is a point estimate, not certainty that p equals 0.8.

In [2]:
y=np.array([1]*8+[0]*2)
print("Point estimate p:",y.mean())

Point estimate p: 0.8


## 2. Interval Estimation

An interval estimator returns lower and upper limits to express uncertainty under a method. Frequentist confidence intervals have repeated-sampling coverage; Bayesian credible intervals assign posterior probability to a parameter under a prior and likelihood. Prediction intervals concern future observations, while tolerance intervals concern population coverage under another design. They answer different questions.

**Example:** a 95% confidence interval for a mean is not an interval containing 95% of individual observations. Its width generally shrinks with independent information; an individual-outcome interval retains outcome variability. Notebook 5.4 derives common confidence intervals and their assumptions.

## 3. Estimator Properties: Bias and Consistency

Bias at parameter θ is $\mathbb E_\theta[\hat\theta]-\theta$. **Unbiasedness** is a finite-sample expectation property, not a statement that each estimate equals truth. **Consistency** means $\hat\theta_n\to\theta$ in probability as sample size grows. An estimator can be biased at every finite n and still be consistent.

**Worked example:** under IID finite-variance sampling, the denominator-n variance $v_n=\sum_i(X_i-\bar X)^2/n$ has expectation $(n-1)\sigma²/n$ and negative bias −σ²/n. The denominator-(n−1) estimator is unbiased. Both are consistent under the usual IID second-moment conditions, although neither repairs selection bias.

Unbiasedness alone does not guarantee consistency: using just X₁ as an estimator of μ is unbiased under an IID model but discards new observations, so its variability need not vanish.

In [3]:
reps,n=12000,5
samples=rng.normal(0,2,size=(reps,n))
biased_variances=samples.var(axis=1,ddof=0)
unbiased_variances=samples.var(axis=1,ddof=1)
print("True variance:",4,"expected denominator-n estimate:",(n-1)/n*4)
print("Simulated averages:",biased_variances.mean(),unbiased_variances.mean())

True variance: 4 expected denominator-n estimate: 3.2
Simulated averages: 3.1991797612359067 3.9989747015448835


## 4. Efficiency

Within a specified comparison class, a more efficient estimator has lower sampling variance for the same target and design; among unbiased estimators this also lowers MSE. “Efficient” requires a context, not an absolute label. Under regularity conditions, the Cramér–Rao bound gives a variance lower bound for unbiased estimators; Module 7 develops it.

Under normal sampling, the sample mean is more variance-efficient than the sample median asymptotically, but the median can be more robust under contamination. Lower variance under the wrong model is not necessarily preferable. Robustness, computational cost, and the estimand matter alongside a theoretical variance comparison.

## 5. Mean Squared Error (MSE)

For a scalar estimator with finite second moment,
$$MSE_\theta(\hat\theta)=\mathbb E_\theta[(\hat\theta-\theta)^2]=\operatorname{Var}_\theta(\hat\theta)+Bias_\theta(\hat\theta)^2.$$
This is an expectation over repeated samples, unlike a realized squared error. A biased shrinkage estimator can reduce MSE by lowering variance enough. The improvement can depend on the true parameter; a shrinkage rule that helps near zero may hurt far from zero.

**Worked example:** observations are Normal(θ,1), n=5, and shrinkage estimator is 0.8 times the sample mean. At θ=1 its variance is 0.64/5=0.128, squared bias 0.04, MSE 0.168, versus sample-mean MSE 0.2. At θ=3 its MSE becomes 0.128+0.36=0.488, worse than 0.2.

In [4]:
n=5
for theta in [1.,3.]:
    variance=.8**2/n;bias=(.8-1)*theta
    print("theta:",theta,"sample mean MSE:",1/n,"shrinkage MSE:",variance+bias*bias)
means=rng.normal(1,1/np.sqrt(n),size=20000)
print("Simulated MSE at theta=1:",np.mean((.8*means-1)**2))

theta: 1.0 sample mean MSE: 0.2 shrinkage MSE: 0.168
theta: 3.0 sample mean MSE: 0.2 shrinkage MSE: 0.4879999999999999
Simulated MSE at theta=1: 0.16891894163004054


## AI/ML application and interpretation

Regularization trades variance for bias in predictive models. Estimator MSE concerns parameter error, while prediction MSE concerns future outcomes and also includes irreducible outcome noise under the model. Do not equate the two simply because both use squared loss. Report the target and repeated-sampling assumptions.

## Practice

1. Distinguish parameter, estimator, and estimate for a sample mean of 12.
2. Can a biased estimator be consistent?
3. An estimator has variance 4 and bias 3. Find MSE.
4. Is X₁ a consistent estimator of μ merely because it is unbiased?
5. Does a lower-variance estimator always have lower MSE?

## Solutions

1. μ is the target, the sample-mean rule is the estimator, and 12 is its observed estimate.
2. Yes; denominator-n sample variance is a standard example under IID finite-variance conditions, with bias vanishing as n grows.
3. 4+3²=13.
4. No. Its distribution does not concentrate as more unused observations arrive.
5. Not if bias differs. Compare variance plus squared bias for the stated parameter and loss.

**Readiness check:** identify the estimand, independent unit, assumptions, uncertainty source, and an important failure mode before using these methods.

**Next:** [Continue studying](4_Confidence-Intervals.ipynb).